In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
def calculate_clearing_price_and_volume(bid_prices, bid_volumes, ask_prices, ask_volumes):
    # Combine all unique prices from bids and asks to evaluate them
    unique_prices = set(bid_prices + ask_prices)
    
    max_volume = 0
    clearing_price = 0
    
    # Iterate through all possible clearing prices
    for price in sorted(unique_prices):
        # Demand: total volume willing to buy at this price or higher
        demand = sum(v for p, v in zip(bid_prices, bid_volumes) if p >= price)
        
        # Supply: total volume willing to sell at this price or lower
        supply = sum(v for p, v in zip(ask_prices, ask_volumes) if p <= price)
        
        # Actual traded volume is constrained by whichever is smaller (supply or demand)
        volume_traded = min(demand, supply)
        
        # Rule 1: Maximize total traded volume
        # Rule 2: Break ties by choosing the higher price
        if volume_traded > max_volume:
            max_volume = volume_traded
            clearing_price = price
        elif volume_traded == max_volume and price > clearing_price:
            clearing_price = price
            
    return clearing_price, max_volume

In [4]:
def calc_optimal_quote(bid_prices, bid_volumes, ask_prices, ask_volumes, buyback_price, fee):
    best_profit = 0
    best_price = 0
    best_volume = 0
    best_fill = 0
    best_clearing_price = 0
    
    # Test a reasonable range of prices and volumes
    # Prices: From 20 to the buyback price (bidding above buyback guarantees a loss)
    test_prices = range(5, buyback_price + 1)
    
    # Volumes: Test from 0 up to 100,000 in increments of 100
    test_volumes = range(0, 100000, 1)
    
    for my_price in test_prices:
        for my_vol in test_volumes:
            # 1. Simulate the clearing price with our order included
            all_bid_p = bid_prices + [my_price]
            all_bid_v = bid_volumes + [my_vol]
    
            clearing_price, max_vol = calculate_clearing_price_and_volume(all_bid_p, all_bid_v, ask_prices, ask_volumes)
                    
            # 2. Calculate our fill based on Priority Rules
            # Rule: We only get filled if our price >= clearing_price
            if my_price < clearing_price:
                my_fill = 0
            else:
                # Total supply available to be distributed at the clearing price
                total_supply = sum(v for p, v in zip(ask_prices, ask_volumes) if p <= clearing_price)
                
                # Build a queue of buyers: (price, is_me, volume)
                buyers = []
                for p, v in zip(bid_prices, bid_volumes):
                    if p >= clearing_price:
                        buyers.append((p, False, v)) # False means it's an existing bot (higher time priority)
                
                buyers.append((my_price, True, my_vol)) # True means it's us (lowest time priority)
                    
                # Sort queue: Highest price first. If tied, 'False' (bots) before 'True' (us)
                buyers.sort(key=lambda x: (-x[0], x[1]))
                
                my_fill = 0
                remaining_supply = total_supply
                
                # Distribute supply down the queue
                for p, is_me, v in buyers:
                    fill = min(v, remaining_supply)
                    if is_me:
                        my_fill = fill
                    remaining_supply -= fill
                    if remaining_supply <= 0:
                        break
            
            # 3. Calculate guaranteed profit
            profit = my_fill * (buyback_price - clearing_price - fee)
            
            if profit > best_profit:
                best_profit = profit
                best_price = my_price
                best_volume = my_vol
                best_fill = my_fill
                best_clearing_price = clearing_price
    
    return {
        "Submit Price": best_price,
        "Submit Volume": best_volume,
        "Expected Fill": best_fill,
        "Clearing Price": best_clearing_price,
        "Guaranteed Profit": best_profit
    }

In [5]:
# Dryland flax

buyback_price = 30
fee = 0  # per unit traded

bid_prices = [30, 29, 28, 27]
bid_volumes = [30000, 5000, 12000, 28000]

ask_prices = [28, 31, 32, 33]
ask_volumes = [40000, 20000, 20000, 30000]

result_dryland = calc_optimal_quote(bid_prices, bid_volumes, ask_prices, ask_volumes, buyback_price, fee)
print("=== Optimal Order ===")
for key, value in result_dryland.items():
    print(f"{key}: {value}")

=== Optimal Order ===
Submit Price: 30
Submit Volume: 9999
Expected Fill: 9999
Clearing Price: 29
Guaranteed Profit: 9999


In [6]:
# Ember mushroom

buyback_price = 20
fee = 0.1  # per unit traded

bid_prices = [20, 19, 18, 17, 16, 15, 14, 13]
bid_volumes = [43000, 17000, 6000, 5000, 10000, 5000, 10000, 7000]

ask_prices = [12, 13, 14, 15, 16, 17, 18, 19]
ask_volumes = [20000, 25000, 35000, 6000, 5000, 0, 10000, 12000]

result_ember = calc_optimal_quote(bid_prices, bid_volumes, ask_prices, ask_volumes, buyback_price, fee)
print("=== Optimal Order ===")
for key, value in result_ember.items():
    print(f"{key}: {value}")

=== Optimal Order ===
Submit Price: 17
Submit Volume: 19999
Expected Fill: 19999
Clearing Price: 16
Guaranteed Profit: 77996.09999999999
